# ClaudeTrader Demo Notebook

This notebook demonstrates the key features and capabilities of ClaudeTrader, an advanced AI-powered trading intelligence system.

## What is ClaudeTrader?

ClaudeTrader combines:
- **Large Language Models (LLMs)** for intelligent reasoning
- **Retrieval-Augmented Generation (RAG)** for contextual knowledge
- **Advanced trading strategies** including SuperTrend, RL, and multi-factor models
- **Natural language interface** for conversational trading insights

## Setup and Installation

In [ ]:
# Install dependencies (uncomment to run)
# !pip install -r requirements.txt

import sys
sys.path.append('..')

from core.engine import ClaudeTrader
from strategies import get_strategy, list_strategies
from utils.data_fetcher import fetch_market_data
from utils.indicators import calculate_rsi, calculate_sma

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
plt.style.use('dark_background')
sns.set_palette('husl')

print("✅ ClaudeTrader modules imported successfully!")

## 1. Initialize ClaudeTrader

In [ ]:
# Initialize ClaudeTrader engine
trader = ClaudeTrader(config_path='../configs/default_config.yaml')

print("🤖 ClaudeTrader initialized and ready!")
print(f"📊 Configuration loaded")
print(f"🧠 LLM: {trader.config['llm']['provider']}/{trader.config['llm']['model']}")
print(f"🔍 RAG: {trader.config['rag']['vector_db']}")

## 2. Conversational Trading Intelligence

Ask ClaudeTrader questions in natural language:

In [ ]:
# Ask a trading question
question = "What is the SuperTrend indicator and how does it work?"

response = trader.query(question)

print(f"❓ Question: {response.query}")
print(f"\n💡 Response:\n{response.response}")
print(f"\n📚 Sources: {', '.join(response.sources)}")
print(f"🎯 Confidence: {response.confidence:.2%}")

In [ ]:
# More questions to try:
questions = [
    "Should I use trend-following or mean-reversion in volatile markets?",
    "What are the best risk management practices for crypto trading?",
    "How do I optimize position sizing based on volatility?"
]

for q in questions:
    response = trader.query(q)
    print(f"\n{'='*80}")
    print(f"Q: {q}")
    print(f"A: {response.response[:200]}...")
    print(f"Confidence: {response.confidence:.2%}")

## 3. Trading Signals Generation

In [ ]:
# Get trading signals for BTC
signals = trader.get_signals(
    symbol='BTC/USD',
    timeframe='1h',
    strategies=['supertrend', 'multi_factor']
)

print(f"📊 Generated {len(signals)} signals for BTC/USD\n")

for i, signal in enumerate(signals, 1):
    print(f"Signal #{i}:")
    print(f"  Action: {signal.action.upper()}")
    print(f"  Strategy: {signal.strategy}")
    print(f"  Price: ${signal.price:.2f}")
    print(f"  Confidence: {signal.confidence:.2%}")
    print(f"  Reasoning: {signal.reasoning}")
    print()

## 4. Strategy Analysis and Backtesting

In [ ]:
# Analyze SuperTrend strategy
analysis = trader.analyze_strategy(
    strategy='supertrend',
    symbol='BTC/USD',
    parameters={'atr_period': 10, 'multiplier': 3.0},
    backtest_period='90d'
)

print("📈 Strategy Analysis: SuperTrend\n")
print(f"Symbol: {analysis['symbol']}")
print(f"Period: {analysis['backtest_period']}")
print(f"\nPerformance Metrics:")

performance = analysis['performance']
for metric, value in performance.items():
    if isinstance(value, float):
        if 'ratio' in metric:
            print(f"  {metric}: {value:.2f}")
        elif 'rate' in metric or 'return' in metric or 'drawdown' in metric:
            print(f"  {metric}: {value:.2%}")
        else:
            print(f"  {metric}: {value:.2f}")
    else:
        print(f"  {metric}: {value}")

print(f"\n🧠 AI Insights:\n{analysis['ai_insights']}")

## 5. Strategy Comparison

In [ ]:
# Compare multiple strategies
comparison = trader.compare_strategies(
    strategies=['supertrend', 'multi_factor', 'hybrid'],
    backtest_period='90d',
    symbol='BTC/USD'
)

print("🏆 Strategy Comparison\n")
print(f"Symbol: {comparison['symbol']}")
print(f"Period: {comparison['backtest_period']}")
print(f"\n💡 Recommendation: {comparison['recommendation']}")

## 6. Available Strategies

In [ ]:
# List all available strategies
strategies = list_strategies()

print("📊 Available Trading Strategies:\n")
for strategy in strategies:
    print(f"  • {strategy}")

## 7. Market Data and Indicators

In [ ]:
# Fetch market data
data = fetch_market_data('BTC/USD', timeframe='1h', limit=100)

# Create DataFrame
df = pd.DataFrame({
    'open': data['open'],
    'high': data['high'],
    'low': data['low'],
    'close': data['close'],
    'volume': data['volume']
})

# Calculate indicators
df['SMA_20'] = calculate_sma(df['close'].tolist(), 20)
df['RSI'] = calculate_rsi(df['close'].tolist(), 14)

print("📊 Market Data Summary:\n")
print(df.describe())

In [ ]:
# Visualize price and indicators
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Price chart
axes[0].plot(df['close'], label='Close Price', linewidth=2)
axes[0].plot(df['SMA_20'], label='SMA 20', linestyle='--', alpha=0.7)
axes[0].set_title('BTC/USD Price Chart', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Price ($)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Volume
axes[1].bar(range(len(df)), df['volume'], alpha=0.6)
axes[1].set_title('Volume', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Volume')
axes[1].grid(True, alpha=0.3)

# RSI
axes[2].plot(df['RSI'], label='RSI', color='orange', linewidth=2)
axes[2].axhline(y=70, color='r', linestyle='--', alpha=0.5, label='Overbought')
axes[2].axhline(y=30, color='g', linestyle='--', alpha=0.5, label='Oversold')
axes[2].set_title('RSI Indicator', fontsize=14, fontweight='bold')
axes[2].set_ylabel('RSI')
axes[2].set_xlabel('Time Period')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Performance Report

In [ ]:
# Generate performance report
report = trader.generate_report(
    period='30d',
    metrics=['sharpe_ratio', 'max_drawdown', 'win_rate']
)

print("📊 Performance Report\n")
print(f"Period: {report['period']}")
print(f"Generated: {report['generated_at']}")
print(f"\nMetrics: {', '.join(report['metrics'])}")
print(f"\nSummary: {report['summary']}")
print(f"\n🧠 AI Insights:\n{report['ai_insights']}")

## 9. Integration with Existing Bots

In [ ]:
# Example: Integrate with existing SuperTrend bot
from integrations.supertrend_bot_integration import SuperTrendBotIntegration

# Initialize integration
integration = SuperTrendBotIntegration()

# Mock signal from existing bot
existing_signal = {
    'symbol': 'BTC/USD',
    'action': 'buy',
    'price': 67234.50,
    'confidence': 0.8,
    'indicators': {
        'supertrend': 65000,
        'atr': 1200
    }
}

# Enhance with AI
enhanced = integration.augment_signals([existing_signal])

print("🔄 Signal Enhancement\n")
print(f"Original Signal: {existing_signal['action'].upper()} @ ${existing_signal['price']}")
print(f"AI Confidence: {enhanced[0]['ai_confidence']:.2%}")
print(f"Risk Level: {enhanced[0]['risk_level']}")
print(f"\nAI Reasoning:\n{enhanced[0]['ai_reasoning']}")

## 10. Custom Strategy Example

In [ ]:
# Get a specific strategy
config = {
    'parameters': {
        'atr_period': 10,
        'multiplier': 3.0
    }
}

supertrend = get_strategy('supertrend', config)

# Generate signal with custom strategy
market_data = {
    'high': df['high'].tolist(),
    'low': df['low'].tolist(),
    'close': df['close'].tolist()
}

signal = supertrend.generate_signal(market_data)

if signal:
    print(f"🎯 Signal Generated:\n")
    print(f"  Action: {signal.action.upper()}")
    print(f"  Price: ${signal.price:.2f}")
    print(f"  Confidence: {signal.confidence:.2%}")
    print(f"  Reasoning: {signal.reasoning}")
    print(f"\n  Indicators:")
    for ind, val in signal.indicators.items():
        print(f"    {ind}: {val:.2f}")

## Conclusion

This notebook demonstrated the core capabilities of ClaudeTrader:

1. ✅ Conversational trading intelligence with LLM
2. ✅ Multi-strategy signal generation
3. ✅ Strategy analysis and backtesting
4. ✅ Technical indicator calculation
5. ✅ Performance reporting
6. ✅ Integration with existing bots

### Next Steps

- Configure API keys for production LLM usage
- Connect to live exchange data feeds
- Customize strategies for your trading style
- Deploy the web dashboard
- Set up automated trading (with caution!)

### Resources

- 📚 [Full Documentation](../docs/API.md)
- 🔗 [GitHub Repository](https://github.com/DaScient/SuperTrendTradingBot)
- 🌐 [DASCIENT Website](https://dascient.com)

---

**Disclaimer**: This is a decision-support tool, not financial advice. Always perform your own research and consult with financial professionals. Trading involves risk of loss.